# Phase 3: feature availability and pipeline costs

Run this after notebook 02 has passed its real-batch correctness checks. This notebook audits
historical feature availability and measures the inference pipeline on small relational batches.
The model still has **random weights and constant features plus relative time**. It does not
train a predictor or establish predictive quality.

1. Save a copy of this notebook to Drive. It uses temporary VM storage and does not mount Drive.
2. Choose **Runtime → Change runtime type → T4 GPU**, save, and connect. For a CPU diagnostic,
   set `DEVICE = "cpu"` in the next cell.
3. In **Secrets**, enable access to the existing read-only `GITHUB_TOKEN` for this private repo.
4. In the next cell, enter the actual unit/hour rate and remaining units shown in Colab's resource
   panel. Leave a value `None` only when the panel does not provide it. Do not invent a value.
5. Run cells in order. If a cell fails, stop the experiment and run the final ZIP-download cell.
   Download the archive before disconnecting the runtime.

The default workload stays at 3 batches of 8 queries, fanouts `[4, 2]`, 64 channels and 10 repeats.
Reuse the data from notebook 02 when this VM still has it. A fresh VM downloads the pinned data;
column projection reduces reading memory, not download size. No Hugging Face token is required.

This notebook and its GPU pipeline remain **unvalidated on the hosted data until you run them**.

In [ ]:
import json
import math
import os
import subprocess
import sys
import tempfile
import uuid
from datetime import datetime, timezone
from pathlib import Path

# Fill these values from the Colab resource panel when they are available.
DEVICE = "cuda"
COMPUTE_UNITS_PER_HOUR = None
COMPUTE_UNITS_REMAINING = None
RESERVE_UNITS = 50
PROJECT_ROOT = Path("/content/temporal-relational-gnn")
DATA_DIR = PROJECT_ROOT / "data/rel-stack"
PROJECT_REF = "main"  # The actual checked-out commit is saved in checkout.log.
REPO_URL = "https://github.com/WeaversMask/temporal-relational-gnn.git"

# Create exportable storage before cloning, installing packages or checking the budget.
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:6]
RUN_DIR = Path("/content/temporary-trgnn-profile/runs") / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
print(f"Run directory: {RUN_DIR}", flush=True)


def run_logged(arguments, log_name, *, cwd=None, env=None):
    with (RUN_DIR / log_name).open("a") as log:
        with subprocess.Popen(
            arguments,
            cwd=PROJECT_ROOT if cwd is None else cwd,
            env=env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        ) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            result = process.wait()
    if result:
        raise subprocess.CalledProcessError(result, arguments)


for name, value in {
    "COMPUTE_UNITS_PER_HOUR": COMPUTE_UNITS_PER_HOUR,
    "COMPUTE_UNITS_REMAINING": COMPUTE_UNITS_REMAINING,
}.items():
    if value is not None and (
        isinstance(value, bool)
        or not isinstance(value, (int, float))
        or not math.isfinite(value)
        or value < 0
    ):
        raise ValueError(f"{name} must be an observed non-negative number or None")

(RUN_DIR / "colab-budget.json").write_text(
    json.dumps(
        {
            "units_per_hour_observed": COMPUTE_UNITS_PER_HOUR,
            "units_remaining_observed": COMPUTE_UNITS_REMAINING,
            "planned_total_units": 200,
            "reserve_units": RESERVE_UNITS,
        },
        indent=2,
        allow_nan=False,
    )
    + "\n"
)
(RUN_DIR / "notebook-run.json").write_text(
    json.dumps(
        {"notebook": "03_audit_profile", "device": DEVICE, "project_ref": PROJECT_REF},
        indent=2,
    )
    + "\n"
)
if DEVICE not in {"cpu", "cuda"}:
    raise ValueError('Set DEVICE to "cpu" or "cuda"')
if COMPUTE_UNITS_REMAINING is not None and COMPUTE_UNITS_REMAINING <= RESERVE_UNITS:
    raise RuntimeError("The 50-unit reserve has been reached. Export the files and disconnect.")
if COMPUTE_UNITS_PER_HOUR is None or COMPUTE_UNITS_REMAINING is None:
    print("Some budget observations are unavailable; the report records them as null.")

Clone or update the project and install its project dependencies. Both steps write logs into the run
folder created above. If either fails, the final export cell is already available.

In [ ]:
from google.colab import userdata

try:
    token = userdata.get("GITHUB_TOKEN")
    with tempfile.TemporaryDirectory() as temporary:
        askpass = Path(temporary) / "askpass.py"
        askpass.write_text(
            "#!/usr/bin/env python3\n"
            "import os, sys\n"
            'print("x-access-token" if "username" in sys.argv[1].lower() '
            'else os.environ["TRGNN_CLONE_TOKEN"])\n'
        )
        askpass.chmod(0o700)
        clone_environment = {
            **os.environ,
            "GIT_ASKPASS": str(askpass),
            "GIT_TERMINAL_PROMPT": "0",
            "TRGNN_CLONE_TOKEN": token,
        }
        if not PROJECT_ROOT.exists():
            command = ["git", "-c", "credential.helper=", "clone", REPO_URL, str(PROJECT_ROOT)]
        else:
            command = [
                "git",
                "-C",
                str(PROJECT_ROOT),
                "-c",
                "credential.helper=",
                "fetch",
                "origin",
            ]
        run_logged(command, "clone.log", cwd=Path("/content"), env=clone_environment)
except Exception as exc:
    (RUN_DIR / "setup-failure.json").write_text(
        json.dumps({"stage": "clone", "error_type": type(exc).__name__}, indent=2) + "\n"
    )
    raise
finally:
    token = None
    if "clone_environment" in globals():
        clone_environment.pop("TRGNN_CLONE_TOKEN", None)

run_logged(["git", "checkout", PROJECT_REF], "checkout.log")
if PROJECT_REF == "main":
    run_logged(["git", "merge", "--ff-only", "origin/main"], "checkout.log")
run_logged(["git", "rev-parse", "HEAD"], "checkout.log")
run_logged(
    [sys.executable, "-m", "pip", "install", "-e", f"{PROJECT_ROOT}[dev,relbench]"],
    "install.log",
)

Check the installed sampler and correctness suite before processing data. Commands run in fresh
Python processes, so they see newly installed packages. If a check fails, export its logs.

In [ ]:
run_logged([sys.executable, "scripts/install_sampler.py"], "install-sampler.log")
arguments = [
    sys.executable,
    "-m",
    "trgnn.preflight",
    "--check-sampler",
    "--output",
    str(RUN_DIR / "preflight.json"),
]
if DEVICE == "cuda":
    arguments.append("--require-cuda")
run_logged(arguments, "preflight.log")
run_logged([sys.executable, "-m", "pytest", "-q"], "correctness.log")

Audit the historical inputs and profile the same small workload across reference, HGAM and blocks.
A matching cached download is reused; otherwise the pinned inputs are downloaded once.

Pipeline totals cover **sampling and feature gathering, transfer, feature/time encoding, GNN and
prediction head**. Downloading and building the graph are one-time preparation and are excluded
from those inference totals. Paired execution modes use the same weights and sampled occurrences.
Stage measurements help locate costs; random-weight timing is still a diagnostic, not evidence
of trained-model quality or a measured final predictor.

No feature choices, model training or larger workload are enabled automatically. Inspect the audit
and timing reports before deciding the next experiment. Do not increase the defaults on this run.

In [ ]:
source_receipt = DATA_DIR / ".trgnn-source.json"
expected_source = json.loads((PROJECT_ROOT / "configs/upstream.json").read_text())["dataset"]
try:
    cached_source = json.loads(source_receipt.read_text())
except (OSError, ValueError):
    cached_source = None

arguments = [
    sys.executable,
    "-m",
    "trgnn.pipeline_profile",
    "--data-dir",
    str(DATA_DIR),
    "--device",
    DEVICE,
    "--batch-size",
    "8",
    "--batches",
    "3",
    "--channels",
    "64",
    "--fanouts",
    "4",
    "2",
    "--repeats",
    "10",
    "--output",
    str(RUN_DIR / "pipeline-profile.json"),
]
if cached_source != expected_source:
    arguments.append("--download")
    print("Downloading the pinned inputs before profiling.", flush=True)
else:
    print("Reusing the pinned local data download.", flush=True)
run_logged(arguments, "pipeline-profile.log")

Inspect the saved reports. A completed run requires a passed audit and completed pipeline
comparisons. Failed or partial runs are diagnostic evidence and should be exported too.

In [ ]:
report_path = RUN_DIR / "pipeline-profile.json"
if report_path.exists():
    report = json.loads(report_path.read_text())
    print("Status:", report["status"])
    print("Scope:", report["scope"])
    print("Trained:", report["trained"], "Includes online pipeline:", report["end_to_end"])
    audit = report.get("feature_audit", {})
    print("Feature audit:", audit.get("status", "not completed"))
    print("Training ready:", audit.get("training_ready", False))
    for error in audit.get("errors", []):
        print("Audit error:", error)
    if "error" in report:
        print("Error:", report["error"])
    for result in report.get("batches", []):
        print("\nBatch", result.get("index"))
        print("Pipeline median ms:", result.get("median_pipeline_ms", {}))
        print("Stage median ms:", json.dumps(result.get("median_stage_ms", {}), indent=2))
        print("Max logit errors:", result.get("max_absolute_logit_error", {}))
    print("Full feature classifications and timing repetitions are in the saved JSON.")
else:
    print("No pipeline report yet; export the available setup reports and logs.")

## Download the evidence

Run this even after clone, installation, preflight or data failures, as long as the first code cell
created the run folder. The ZIP contains the available reports and logs, excluding the database.
Keep the downloaded archive, then **disconnect and delete the runtime** to stop GPU use.
If a failure prevents profiling, do not continue with model training or larger settings.

In [ ]:
import shutil

from google.colab import files

archive = shutil.make_archive(
    "/content/trgnn-profile-results-" + run_id, "zip", root_dir=str(RUN_DIR)
)
files.download(archive)